# Drug–Gene Interaction Network Analysis

This notebook analyzes the **ChG-InterDecagon** drug–gene interaction dataset
from the [SNAP Biodata repository](https://snap.stanford.edu/biodata/datasets/10016/)
as a **bipartite graph**:

- **Drugs** (PubChem CIDs) and **Genes** are the two node types.
- An edge connects a drug to a gene whenever the drug is known to target/interact
  with that gene.

The notebook is organized into four stages:

1. **Data acquisition** – download and inspect the raw edge list.
2. **Graph construction** – build the bipartite NetworkX graph `B`.
3. **Function definitions** – all analysis/visualization/community-detection
   logic, grouped by purpose.
4. **Running the analysis** – the functions are actually *called* here, split
   into clearly labeled sections (metrics, visualizations, community
   detection, node-level exploration).


## 0. Imports

All third-party and standard-library imports live here so dependencies are
visible at a glance. (`requests`, `time`, and `matplotlib.cm` from the
original notebook were never actually used, so they were dropped.)

In [2]:
# --- Standard library ---
import os
import csv
import gzip
import math
import shutil
import random
import collections
import urllib.request
from statistics import mean

# --- Data handling ---
import numpy as np
import pandas as pd

# --- Graph analysis & visualization ---
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# --- Community detection (Leiden algorithm) ---
import igraph as ig
import leidenalg as la


## 1. Data Acquisition

Download the compressed edge-list file from SNAP and decompress it locally
(the download is skipped automatically if the file already exists). We then
peek at the first few lines and let Python's CSV `Sniffer` detect the
delimiter and whether a header row is present, so we know how to parse it
correctly with pandas.


In [3]:
# Provided by tutoring staff (kept as-is, only docstring/comments cleaned up)

def download_and_extract(url, out_path):
    """
    Download a gzip-compressed file and decompress it to ``out_path``.

    If ``out_path`` already exists, the download is skipped entirely.

    Args:
        url (str): URL of the ``.gz`` file to download.
        out_path (str): Destination path for the decompressed file.

    Returns:
        str: The path to the decompressed file (same as ``out_path``).
    """
    if os.path.exists(out_path):
        print(f"'{out_path}' is already here - skipping download.")
        return out_path

    gz_path = out_path + ".gz"
    print(f"Downloading {url}")

    # Some servers reject the default urllib user agent, so we pretend to be a browser.
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request) as response, open(gz_path, "wb") as f:
        shutil.copyfileobj(response, f)

    print("Decompressing...")
    with gzip.open(gz_path, "rb") as f_in, open(out_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(gz_path)  # drop the .gz, keep only the plain file

    print("Ready:", os.path.abspath(out_path))
    return out_path


print("Working folder:", os.getcwd())


Working folder: /home/belugaboy/Downloads


In [4]:
# Download the ChG-InterDecagon drug-gene target edge list
path = download_and_extract(
    "https://snap.stanford.edu/biodata/datasets/10016/files/ChG-InterDecagon_targets.csv.gz",
    "ChG-InterDecagon_targets.csv",
)

# --- Quick sanity check on the raw file ---
print("--- first 10 lines ---")
with open(path, "r") as f:
    for _ in range(10):
        print(f.readline().rstrip())

# Let Python's Sniffer guess the delimiter/header from a small sample
with open(path, "r") as f:
    sample = f.read(2048)

dialect = csv.Sniffer().sniff(sample)
has_header = csv.Sniffer().has_header(sample)
print(f"\nDetected delimiter: {dialect.delimiter!r}")
print("Header row present:", has_header)


'ChG-InterDecagon_targets.csv' is already here - skipping download.
--- first 10 lines ---
# Drug	Gene
CID000060752,3757
CID006918155,2908
CID103052762,3359
CID023668479,1230
CID000028864,1269
CID000028864,124274
CID000028864,2849
CID000028864,2847
CID000028864,2844

Detected delimiter: 'D'
Header row present: False


## 2. Building the Bipartite Graph

We read the edge list with pandas (skipping commented header lines) and use
it to build a bipartite `networkx.Graph`, where drug nodes and gene nodes are
tagged with a `bipartite`/`type` attribute so they can be told apart later.


In [5]:
# Read the edge list (commented lines starting with '#' are skipped)
df = pd.read_csv(path, comment='#', header=None, names=['drug', 'gene'], dtype=str)

# Build the bipartite graph: drugs on one side, genes on the other
B = nx.Graph()
drugs = df['drug'].unique()
genes = df['gene'].unique()
B.add_nodes_from(drugs, bipartite=0, type='drug')
B.add_nodes_from(genes, bipartite=1, type='gene')
B.add_edges_from(df[['drug', 'gene']].itertuples(index=False, name=None))

print(f"Bipartite graph created: {B.number_of_nodes()} nodes, {B.number_of_edges()} edges")


Bipartite graph created: 9569 nodes, 131034 edges


## 3. Function Definitions

All reusable logic is defined here, grouped by what it's used for. Nothing in this section produces output by itself — the functions are actually *run* in section 4.

### 3.1 Network Metrics & Structure

Basic descriptive statistics about the graph: degree averages, density, clustering, and connected components.

In [6]:
def calculate_network_metrics(B, drug_list=None, gene_list=None):
    """
    Calculate basic descriptive metrics for a bipartite graph.

    Computes average degree (overall and per partition), bipartite density,
    and average clustering coefficient.

    Args:
        B (networkx.Graph): The bipartite graph to analyze.
        drug_list (list, optional): Nodes belonging to the "drug" partition.
            If None, inferred from the graph's bipartite metadata.
        gene_list (list, optional): Nodes belonging to the "gene" partition.
            If None, inferred from the graph's bipartite metadata.

    Returns:
        dict: Keys are "avg_degree", "avg_top_degree", "avg_bottom_degree",
        "density", and "avg_clustering".
    """
    # Infer partitions from graph metadata if not provided explicitly
    if drug_list is None or gene_list is None:
        top_nodes, bottom_nodes = nx.bipartite.sets(B)
    else:
        top_nodes = set(drug_list) & set(B.nodes())
        bottom_nodes = set(gene_list) & set(B.nodes())

    # Degrees per partition
    degree_map = dict(nx.degree(B))
    top_degrees = {n: degree_map[n] for n in top_nodes}
    bottom_degrees = {n: degree_map[n] for n in bottom_nodes}

    avg_degree = mean(degree_map.values()) if degree_map else 0.0
    avg_top_degree = mean(top_degrees.values()) if top_degrees else 0.0
    avg_bottom_degree = mean(bottom_degrees.values()) if bottom_degrees else 0.0

    density = nx.bipartite.density(B, top_nodes)
    avg_clustering = nx.bipartite.average_clustering(B, mode="min")

    print("--- Network Metrics (Bipartite) ---")
    print(f"Partition sizes: drugs={len(top_nodes)}, genes={len(bottom_nodes)}")
    print("Average Degree (all nodes):", avg_degree)
    print("Average Degree (drugs)    :", avg_top_degree)
    print("Average Degree (genes)    :", avg_bottom_degree)
    print("Bipartite Density         :", density)
    print("Average Clustering        :", avg_clustering)
    print("-----------------------------------")

    return {
        "avg_degree": avg_degree,
        "avg_top_degree": avg_top_degree,
        "avg_bottom_degree": avg_bottom_degree,
        "density": density,
        "avg_clustering": avg_clustering,
    }


In [7]:
def analyze_components(B):
    """
    Analyze the connected components of an undirected bipartite graph.

    Args:
        B (networkx.Graph): The bipartite graph to analyze.

    Returns:
        dict: Keys are "components" (list of node sets), "num_components",
        "largest_comp_size", and "largest_comp" (the largest component's
        node set, handy for subgraph extraction).
    """
    components = list(nx.connected_components(B))

    num_components = len(components)
    largest_comp = max(components, key=len)
    largest_comp_size = len(largest_comp)

    print("--- Connected Components ---")
    print("Number of components      :", num_components)
    print("Largest component (nodes) :", largest_comp_size)
    print("---------------------------")

    return {
        "components": components,
        "num_components": num_components,
        "largest_comp_size": largest_comp_size,
        "largest_comp": largest_comp,
    }


In [8]:
def centrality_measures(G, k_samples=300, verbose=False):
    """
    Compute Betweenness, Closeness, and Eigenvector centrality for a graph.

    Betweenness centrality is approximated via sampling (``k_samples`` source
    nodes) since exact betweenness is expensive on large graphs.

    Args:
        G (networkx.Graph): The graph to analyze.
        k_samples (int): Number of source nodes to sample for the
            betweenness-centrality approximation. Larger values are more
            accurate but slower (k=300 takes roughly 2 minutes on this
            dataset).
        verbose (bool): If True, also print the full per-node centrality
            dictionaries (very noisy for large graphs). Default False.

    Returns:
        tuple: (betweenness, closeness, eigenvector) centrality dicts,
        each mapping node -> centrality score. If eigenvector centrality
        fails to converge, that dict maps every node to None.
    """
    print("Computing Betweenness Centrality (Approximated)...")
    betweenness = nx.betweenness_centrality(G, k=k_samples)
    print("Average Betweenness Centrality:", sum(betweenness.values()) / len(betweenness))
    if verbose:
        print(betweenness)

    print("Computing Closeness Centrality...")
    closeness = nx.closeness_centrality(G)
    print("Average Closeness Centrality:", sum(closeness.values()) / len(closeness))
    if verbose:
        print(closeness)

    print("Computing Eigenvector Centrality...")
    try:
        eigenvector = nx.eigenvector_centrality(G, max_iter=1000)
        print("Average Eigenvector Centrality:", sum(eigenvector.values()) / len(eigenvector))
        if verbose:
            print(eigenvector)
    except nx.PowerIterationFailedConvergence:
        print("Warning: Eigenvector centrality failed to converge. Returning None for all nodes.")
        eigenvector = {node: None for node in G.nodes()}

    return betweenness, closeness, eigenvector


### 3.2 Visualization Functions

Functions for plotting the graph itself — sampled subgraphs, connected components, and degree distributions.

In [9]:
def plot_sampled_subgraph(B, drugs, genes, n_drugs=100, n_genes=100, seed=None):
    """
    Plot a random sample of drug and gene nodes with drugs on the left and
    genes on the right, to get a quick visual feel for the graph's density
    without rendering all nodes (which would be unreadable).

    Args:
        B (networkx.Graph): The bipartite graph.
        drugs (array-like): All drug node identifiers.
        genes (array-like): All gene node identifiers.
        n_drugs (int): Number of drug nodes to sample.
        n_genes (int): Number of gene nodes to sample.
        seed (int, optional): Random seed for reproducible sampling.
    """
    if seed is not None:
        random.seed(seed)

    sample_drugs = random.sample(list(drugs), n_drugs)
    sample_genes = random.sample(list(genes), n_genes)
    sample_nodes = sample_drugs + sample_genes

    subgraph = B.subgraph(sample_nodes)
    drug_set = set(drugs)
    node_color = ['#e05c5c' if n in drug_set else '#5c9ee0' for n in subgraph.nodes()]

    # Manual two-column layout: drugs jittered around x=0, genes around x=n_genes
    spread = max(n_drugs, n_genes)
    pos = {}
    pos.update((n, (random.randrange(-spread // 10, spread // 10 + 1), random.randrange(-spread, spread)))
               for n in sample_drugs)
    pos.update((n, (spread + random.randrange(-spread // 10, spread // 10 + 1), random.randrange(-spread, spread)))
               for n in sample_genes)

    nx.draw(subgraph, pos=pos, with_labels=False, node_size=40,
            node_color=node_color, edge_color='gray', alpha=0.7)
    plt.title(f'Specified Sample of ChG-InterDecagon\n(N={len(sample_nodes)} nodes)',
              fontsize=14, fontweight='bold')

    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#e05c5c', markersize=8, label='Drugs'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#5c9ee0', markersize=8, label='Genes'),
    ]
    plt.legend(handles=legend_elements, loc='lower center')
    plt.show()


In [10]:
def visualize_components(B, drug_list, gene_list, max_components=10):
    """
    Visualize the largest connected components of a bipartite graph as a
    grid of subplots, coloring drugs and genes differently.

    Args:
        B (networkx.Graph): The bipartite graph.
        drug_list (list): Drug node identifiers.
        gene_list (list): Gene node identifiers.
        max_components (int): Maximum number of components to plot,
            selected by descending size.
    """
    drug_set = set(drug_list)
    gene_set = set(gene_list)

    # Sort components by size, largest first, and keep only the top N
    components = sorted(nx.connected_components(B), key=len, reverse=True)
    components_to_plot = components[:max_components]

    n = len(components_to_plot)
    ncols = 3
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes = axes.flatten() if n > 1 else [axes]

    for i, comp in enumerate(components_to_plot):
        ax = axes[i]
        subgraph = B.subgraph(comp)

        top = [node for node in comp if node in drug_set]
        node_colors = ["#e05c5c" if node in drug_set else "#5c9ee0" for node in subgraph.nodes()]

        # Bipartite layout: drugs on one side, genes on the other
        pos = nx.bipartite_layout(subgraph, top, align="horizontal")

        nx.draw(
            subgraph, pos,
            ax=ax,
            node_color=node_colors,
            node_shape="o",
            node_size=2500,
            edge_color="#cccccc",
            with_labels=len(comp) <= 30,  # labels only for small components (readability)
            font_size=7,
        )

    # Hide any unused subplot axes
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    # Shared legend for the whole figure
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor="#e05c5c", markersize=20, label="Drug"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="#5c9ee0", markersize=20, label="Gene"),
    ]
    fig.legend(handles=legend_elements, loc="lower right", ncol=2, fontsize=24)
    fig.suptitle(
        f"Connected Components of Bipartite Graph  (showing {n} of {len(components)})",
        fontsize=33, fontweight="bold", y=1.01
    )

    plt.tight_layout()
    plt.show()


In [11]:
def degree_distribution_log_log(graph, title='Degree Distribution (Log-Log)'):
    """
    Plot the overall degree distribution of a graph on log-log axes.

    Args:
        graph (networkx.Graph): The graph to analyze.
        title (str): Plot title.
    """
    degrees = [d for _, d in graph.degree()]

    # Count how many nodes have each degree value
    degree_counts = collections.Counter(degrees)
    deg, cnt = zip(*degree_counts.items())

    # Sort by degree so the plotted points read left-to-right in order
    deg = np.array(deg)
    cnt = np.array(cnt)
    idx = np.argsort(deg)
    deg, cnt = deg[idx], cnt[idx]

    plt.figure(figsize=(10, 6))
    plt.loglog(deg, cnt, 'o', color='blue', alpha=0.7)
    plt.title(title)
    plt.xlabel('Degree (k)')
    plt.ylabel('Number of Nodes (N(k))')
    plt.grid(True, which="both", ls="-", alpha=0.2)
    plt.tight_layout()
    plt.show()


def plot_degree_distribution(graph, drug_nodes, gene_nodes):
    """
    Plot the degree probability distribution P(k) separately for drug nodes
    and gene nodes, side by side on log-log axes.

    Args:
        graph (networkx.Graph): The bipartite graph.
        drug_nodes (list): Drug node identifiers.
        gene_nodes (list): Gene node identifiers.
    """
    all_degrees = dict(graph.degree())
    drug_degrees = [all_degrees[node] for node in drug_nodes if node in all_degrees]
    gene_degrees = [all_degrees[node] for node in gene_nodes if node in all_degrees]

    def get_degree_distribution(degrees):
        """Return (sorted degree values, corresponding P(k) probabilities)."""
        degree_counts = collections.Counter(degrees)
        total_nodes = len(degrees)
        sorted_degrees = sorted(degree_counts.keys())
        probabilities = [degree_counts[k] / total_nodes for k in sorted_degrees]
        return sorted_degrees, probabilities

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 8))

    drug_k, drug_pk = get_degree_distribution(drug_degrees)
    ax1.scatter(drug_k, drug_pk, color='skyblue', edgecolor='black', alpha=0.8, s=80, label='Data')
    ax1.set_xscale('log')
    ax1.set_yscale('log')
    ax1.set_title(f'Log-Log Degree Distribution of Drug Nodes (n={len(drug_degrees)})', fontsize=14)
    ax1.set_xlabel('Degree (k)', fontsize=12)
    ax1.set_ylabel('Probability P(k)', fontsize=12)
    ax1.grid(True, which="both", ls="--", alpha=0.5)

    gene_k, gene_pk = get_degree_distribution(gene_degrees)
    ax2.scatter(gene_k, gene_pk, color='lightgreen', edgecolor='black', alpha=0.8, s=80, label='Data')
    ax2.set_xscale('log')
    ax2.set_yscale('log')
    ax2.set_title(f'Log-Log Degree Distribution of Gene Nodes (n={len(gene_degrees)})', fontsize=14)
    ax2.set_xlabel('Degree (k)', fontsize=12)
    ax2.set_ylabel('Probability P(k)', fontsize=12)
    ax2.grid(True, which="both", ls="--", alpha=0.5)

    plt.tight_layout()
    plt.show()


### 3.3 Community Detection (Leiden Clustering)

We use the Leiden algorithm to partition the graph into communities, then
collapse each community into a single node to form a **quotient graph** —
a birds-eye view of how communities connect to one another.


In [12]:
def make_quotient_graph(G, partition):
    """
    Collapse a graph into a quotient graph where each node is a community
    (from ``partition``) and edge weights count the number of original
    edges between each pair of communities.

    Args:
        G (networkx.Graph): The original graph.
        partition (dict): Mapping of node -> community id.

    Returns:
        networkx.Graph: The quotient (community-level) graph. Edge weights
        are stored under the "weight" attribute.
    """
    Q = nx.Graph()
    Q.add_nodes_from(set(partition.values()))

    for u, v in G.edges():
        c_u, c_v = partition[u], partition[v]
        if c_u != c_v:
            if Q.has_edge(c_u, c_v):
                Q[c_u][c_v]['weight'] += 1
            else:
                Q.add_edge(c_u, c_v, weight=1)
    return Q


def run_leiden_partition(B):
    """
    Run Leiden community detection on a NetworkX graph and build the
    corresponding quotient graph.

    Args:
        B (networkx.Graph): The graph to partition.

    Returns:
        tuple: (partition_dict, Q, community_sizes)
            partition_dict (dict): node -> community id.
            Q (networkx.Graph): quotient graph of communities.
            community_sizes (dict): community id -> number of member nodes.
    """
    G_ig = ig.Graph.from_networkx(B)
    partition = la.find_partition(G_ig, la.ModularityVertexPartition)
    membership = partition.membership

    nx_nodes = list(B.nodes())
    partition_dict = {nx_nodes[i]: membership[i] for i in range(len(nx_nodes))}

    Q = make_quotient_graph(B, partition_dict)
    community_sizes = {c: sum(1 for v in partition_dict.values() if v == c) for c in Q.nodes()}

    return partition_dict, Q, community_sizes


In [13]:
def plot_quotient_graph(Q, community_sizes, title='Quotient Graph (communities as nodes)',
                         node_size_scale=300, show_labels=False, label_font_size=7,
                         show_edge_labels=False):
    """
    Draw a quotient graph, with node size scaled (log) by community size and
    edge width scaled by the number of inter-community edges.

    Args:
        Q (networkx.Graph): The quotient graph (see ``make_quotient_graph``).
        community_sizes (dict): community id -> number of member nodes.
        title (str): Plot title.
        node_size_scale (float): Base multiplier for node size.
        show_labels (bool): If True, label each node with its community id
            and size (best for a small number of communities).
        label_font_size (int): Font size used for node labels.
        show_edge_labels (bool): If True, annotate edges with their weight.
    """
    fig, ax = plt.subplots(figsize=(12, 10))
    pos = nx.circular_layout(Q)

    # Log-scaled node sizes so a few giant communities don't swamp the layout
    node_sizes = [node_size_scale * np.log1p(community_sizes[c]) for c in Q.nodes()]

    cmap = plt.get_cmap('tab20', max(len(Q.nodes()), 1))
    colors = [cmap(i) for i, _ in enumerate(Q.nodes())]

    edge_weights = [Q[u][v]['weight'] for u, v in Q.edges()]
    max_w = max(edge_weights) if edge_weights else 1
    edge_widths = [3 * w / max_w for w in edge_weights]

    nx.draw_networkx_nodes(Q, pos, node_size=node_sizes, node_color=colors, alpha=0.9, ax=ax)
    nx.draw_networkx_edges(Q, pos, width=edge_widths, alpha=0.7, edge_color='black', ax=ax)

    if show_labels:
        nx.draw_networkx_labels(
            Q, pos,
            labels={c: f'C{c}\n(n={community_sizes[c]})' for c in Q.nodes()},
            font_size=label_font_size, ax=ax,
        )
    if show_edge_labels:
        nx.draw_networkx_edge_labels(
            Q, pos,
            edge_labels={(u, v): Q[u][v]['weight'] for u, v in Q.edges()},
            font_size=10, ax=ax,
        )

    ax.set_title(title, fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


def filter_communities_by_size(Q, community_sizes, max_size=30, keep='small'):
    """
    Return a subgraph of the quotient graph containing only communities
    above or below a size threshold.

    Args:
        Q (networkx.Graph): The quotient graph.
        community_sizes (dict): community id -> number of member nodes.
        max_size (int): Size threshold ``M``.
        keep (str): "small" keeps communities with size <= max_size,
            "big" keeps communities with size > max_size.

    Returns:
        networkx.Graph: The filtered subgraph (a view onto ``Q``).
    """
    if keep == 'big':
        significant = {c for c, s in community_sizes.items() if s > max_size}
    elif keep == 'small':
        significant = {c for c, s in community_sizes.items() if s <= max_size}
    else:
        raise ValueError("keep must be 'small' or 'big'")

    return Q.subgraph(significant)


### 3.4 Node-Level Exploration

Tools for zooming in on individual nodes: comparing two nodes' neighborhoods,
and visualizing a single node together with its immediate neighbors.

> **Note:** `display_node_and_neighbors` optionally decodes raw drug/gene IDs
> into human-readable names via `load_annotations()` and `decode_drug_cids()`.
> Those two helper functions are **not defined in this notebook** — they must
> come from an external annotation-loading step. The function below falls
> back to showing raw IDs if they aren't available, so it still works without
> them.


In [ ]:
def compare_node_neighborhoods(graph, node1, node2):
    """
    Compare the neighborhoods of two nodes: shared neighbors, neighbors
    unique to each, and Jaccard similarity.

    Args:
        graph (networkx.Graph): The graph containing both nodes.
        node1: First node identifier.
        node2: Second node identifier.
    """
    if not graph.has_node(node1) or not graph.has_node(node2):
        print(f"\nError: One or both nodes ({node1}, {node2}) do not exist.")
        return

    set1 = set(graph.neighbors(node1))
    set2 = set(graph.neighbors(node2))

    deg1, deg2 = len(set1), len(set2)
    shared = set1 & set2
    total_union = set1 | set2

    jaccard = (len(shared) / len(total_union)) * 100 if total_union else 0.0
    pct_of_1_shared = (len(shared) / deg1) * 100 if deg1 else 0.0
    pct_of_2_shared = (len(shared) / deg2) * 100 if deg2 else 0.0

    print(f"\n=== Neighborhood Comparison: {node1} vs {node2} ===")
    print(f"{'Metric':<30} | {'Node 1 (' + node1 + ')':<25} | {'Node 2 (' + node2 + ')':<25}")
    print("-" * 88)
    print(f"{'Total Degree':<30} | {deg1:<25} | {deg2:<25}")
    print(f"{'% of own neighbors shared':<30} | {pct_of_1_shared:.2f}%{'':<24} | {pct_of_2_shared:.2f}%")
    print(f"Overall Neighborhood Similarity (Jaccard Index): {jaccard:.2f}%\n")

    def print_sample(title, data_set, sample_size=10):
        """Print up to ``sample_size`` items from a set, noting how many more exist."""
        data_list = sorted(data_set)
        print(f"--- {title} (Total: {len(data_list)}) ---")
        if not data_list:
            print("  None")
        elif len(data_list) <= sample_size:
            print(f"  {', '.join(data_list)}")
        else:
            print(f"  {', '.join(data_list[:sample_size])} ... (+ {len(data_list) - sample_size} more)")
        print()

    print_sample("Shared Neighbors", shared)
    print_sample(f"Unique to {node1}", set1 - set2)
    print_sample(f"Unique to {node2}", set2 - set1)


def display_node_and_neighbors(graph, node_id):
    """
    Print and visualize a node together with its immediate neighbors,
    decoding drug/gene IDs into readable names when annotation-lookup
    helpers are available.

    Args:
        graph (networkx.Graph): The graph containing the node.
        node_id: The central node's identifier.
    """
    if not graph.has_node(node_id):
        print(f"Error: Node '{node_id}' not found in the graph.")
        return

    neighbors = list(graph.neighbors(node_id))

    # Try to load human-readable annotations; fall back to raw IDs if the
    # loader functions aren't defined elsewhere in the environment.
    try:
        gene_name_map, drug_name_map = load_annotations()

        def decode(node):
            if str(node).startswith('CID'):
                return decode_drug_cids([node], drug_name_map).get(node, node)
            return gene_name_map.get(str(node), node)
    except NameError:
        print("Note: load_annotations()/decode_drug_cids() not available - showing raw IDs.")

        def decode(node):
            return node

    labels = {node_id: decode(node_id)}
    decoded_neighbors_display = []
    for neighbor in neighbors:
        decoded = decode(neighbor)
        labels[neighbor] = decoded
        decoded_neighbors_display.append(decoded)

    print(f"\n--- Neighbors for Node: {labels[node_id]} ---")
    print(f"Central Node: {labels[node_id]}")
    print(f"Immediate Neighbors ({len(neighbors)}): {', '.join(decoded_neighbors_display)}")
    print("--------------------------------------")

    # Visualize the central node together with its immediate neighbors
    nodes_to_draw = [node_id] + neighbors
    subgraph = graph.subgraph(nodes_to_draw)

    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(subgraph, k=0.5, iterations=50, seed=42)

    node_colors = ['red' if node == node_id else 'turquoise' for node in subgraph.nodes()]
    nx.draw_networkx_nodes(subgraph, pos, node_color=node_colors, node_size=2000)
    nx.draw_networkx_edges(subgraph, pos, width=1.0, alpha=0.5, edge_color='gray')
    nx.draw_networkx_labels(subgraph, pos, labels=labels, font_size=9, font_color='black')

    plt.title(f'Node: {labels[node_id]} and its Immediate Neighbors', size=15)
    plt.axis('off')
    plt.show()


## 4. Running the Analysis

With every function defined above, we now actually call them, grouped by
what kind of question they answer.


### 4.1 Network Metrics & Structure

Overall size/shape of the network, connected components, and centrality.

In [ ]:
metrics = calculate_network_metrics(B, drugs, genes)
component_info = analyze_components(B)
b_cent, c_cent, e_cent = centrality_measures(B)


--- Network Metrics (Bipartite) ---
Partition sizes: drugs=1774, genes=7795
Average Degree (all nodes): 27.38718779391786
Average Degree (drugs)    : 73.86358511837655
Average Degree (genes)    : 16.810006414368186
Bipartite Density         : 0.009475764607873836
Average Clustering        : 0.6694130855151152
-----------------------------------
--- Connected Components ---
Number of components      : 8
Largest component (nodes) : 9538
---------------------------
Computing Betweenness Centrality (Approximated)...
Average Betweenness Centrality: 0.0002040250691447891
Computing Closeness Centrality...


### 4.2 Visualizations

A sampled overview, the connected components, and degree distributions.

In [ ]:
plot_sampled_subgraph(B, drugs, genes, n_drugs=100, n_genes=100, seed=0)


In [ ]:
visualize_components(B, drugs, genes)


In [ ]:
degree_distribution_log_log(B, title='Degree Distribution of Network B (Log-Log)')
plot_degree_distribution(B, drugs, genes)


### 4.3 Community Detection

Run Leiden clustering, view the full quotient graph, then zoom in on the
smaller communities (size ≤ 30) with labels for readability.


In [ ]:
partition_dict, Q, community_sizes = run_leiden_partition(B)


In [ ]:
# Full quotient graph, unlabeled (too many communities to label legibly)
plot_quotient_graph(Q, community_sizes, node_size_scale=300, show_labels=False)


In [ ]:
# Zoom in on the smaller communities (<= 30 members), with labels this time
Q_small = filter_communities_by_size(Q, community_sizes, max_size=30, keep='small')
plot_quotient_graph(
    Q_small, community_sizes,
    title='Quotient Graph – Small Communities (size <= 30)',
    node_size_scale=550, show_labels=True, label_font_size=12, show_edge_labels=True,
)


### 4.4 Node-Level Exploration

Compare two specific drug nodes' neighborhoods, then inspect one node and
its immediate neighbors. (`display_node_and_neighbors` will show raw IDs if
no annotation lookup is available — see the note in section 3.4. Both calls
below were originally written against an undefined variable `G`; that has
been corrected to `B`, the graph actually built in this notebook.)


In [ ]:
compare_node_neighborhoods(B, 'CID000000753', 'CID100000753')


In [ ]:
display_node_and_neighbors(B, '131')
